# Part 2 — Advanced Git Commands

## Module 04: Essential Git Commands

This notebook covers topics 08–13: removing and renaming files, restoring changes, resetting commits, reverting commits, and cleaning untracked files.

> **Important:** Practice these commands in a disposable test repository. Commands such as `git reset --hard` and `git clean` can permanently discard work. Inspect the target and make a backup before using them.

## Learning Objectives

- Remove tracked files using `git rm`.
- Rename or move tracked files using `git mv`.
- Discard working-tree changes or unstage files using `git restore`.
- Understand the effects of `git reset` modes.
- Undo a commit safely with `git revert`.
- Preview and remove untracked files with `git clean`.

## Prerequisites

You should understand the working directory, staging area, commits, and basic Git history.

Check your repository before practising:

```powershell
git status
```


---

# 08. Git RM

## A. What is `git rm`?

`git rm` removes a file from the working directory and stages that deletion for the next commit. It is used for files that Git is already tracking.

## B. Syntax

```bash
git rm <file>
```

To remove a file from Git tracking but keep it on disk:

```bash
git rm --cached <file>
```

## C. Practical Example

Assume `old-notes.txt` is a tracked file:

```bash
git status
git rm old-notes.txt
git status
git commit -m "Remove old notes file"
```

## D. Expected Result

- `git rm` removes `old-notes.txt` from the working directory.
- The deletion appears in the staging area.
- The commit records the deletion in repository history.

The exact status text may vary by Git version and repository state.

## E. Important Difference

| Command | Result |
|---|---|
| `git rm file.txt` | Deletes file and stages deletion |
| `git rm --cached file.txt` | Keeps file locally, stages removal from tracking |
| `Remove-Item file.txt` | Deletes file from disk; Git detects the deletion, but it is not staged automatically |

## F. Common Mistake

Do not use `git rm` when you only want to stop tracking a file but keep your local copy. Use `git rm --cached` for that purpose.

## G. Interview Question

**Q: What is the difference between `git rm` and `git rm --cached`?**

A: `git rm` removes the file from both the working directory and Git's index. `git rm --cached` removes it from the index but leaves the local file on disk.

## H. Exercise

1. Create and commit a sample file.
2. Run `git rm` on that file.
3. Check `git status`.
4. Commit the deletion.
5. Practise `git rm --cached` on a separate sample file.


---

# 09. Git MV

## A. What is `git mv`?

`git mv` renames or moves a tracked file and stages the change. Git ultimately records file content and paths; it detects renames when comparing commits.

## B. Syntax

```bash
git mv <old-path> <new-path>
```

## C. Practical Example

```bash
git mv draft.txt final-draft.txt
git status
git commit -m "Rename draft file"
```

To move a file into another directory:

```bash
git mv final-draft.txt documents/final-draft.txt
```

The destination directory must already exist.

## D. Expected Result

- The file appears at the new path.
- The rename is staged.
- The next commit records the path change.

## E. Alternative Method

You can also rename a file using File Explorer or `Move-Item`, then stage the change with:

```bash
git add -A
```

## F. Interview Question

**Q: Does Git store a separate rename operation?**

A: Git tracks snapshots of file content and paths. Rename detection is inferred when comparing snapshots.

## G. Exercise

1. Create and commit `sample.txt`.
2. Rename it to `renamed-sample.txt` using `git mv`.
3. Inspect `git status`.
4. Commit the rename.


---

# 10. Git Restore

## A. What is `git restore`?

`git restore` restores file contents from a source such as the index or a commit. It can discard unstaged working-tree changes or unstage staged changes.

## B. Common Commands

Discard unstaged changes in a file:

```bash
git restore <file>
```

Unstage a file while keeping its working-tree changes:

```bash
git restore --staged <file>
```

Restore a file from a specific commit:

```bash
git restore --source=<commit-id> <file>
```

## C. Practical Example

1. Modify a tracked file without staging it.
2. Run:

```bash
git diff
git restore notes.txt
git status
```

The unstaged edits to `notes.txt` are discarded and the file is restored from the index.

Now stage a change and unstage it:

```bash
git add notes.txt
git restore --staged notes.txt
git status
```

The file remains modified in the working directory but is no longer staged.

## D. Important Warning

`git restore <file>` can overwrite uncommitted edits in that file. Review the file and use `git diff` before restoring.

## E. Interview Question

**Q: How do you unstage a file without deleting its changes?**

A: Use `git restore --staged <file>`.

## F. Exercise

Practise both cases separately: discard an unstaged change, then unstage a staged change while keeping its contents.


---

# 11. Git Reset

## A. What is `git reset`?

`git reset` moves the current branch reference to another commit. Depending on the selected mode, it may also change the staging area and working directory.

## B. Reset Modes

| Mode | Branch / HEAD | Staging area | Working directory |
|---|---|---|---|
| `--soft` | Moves | Preserved | Preserved |
| `--mixed` (default) | Moves | Reset to target | Preserved |
| `--hard` | Moves | Reset to target | Reset to target |

## C. Common Syntax

```bash
git reset --soft HEAD~1
git reset --mixed HEAD~1
git reset --hard HEAD~1
```

`HEAD~1` means the first parent of the current commit.

## D. Practical Examples

Use a disposable repository with at least two commits.

Keep the latest commit's changes staged, but remove the commit:

```bash
git reset --soft HEAD~1
git status
```

Remove the latest commit and unstage its changes, keeping file contents:

```bash
git reset --mixed HEAD~1
git status
```

Move the branch back and discard tracked working-tree changes:

```bash
git reset --hard HEAD~1
```

## E. Safety Warning

`git reset --hard` can permanently discard uncommitted changes and make commits unreachable from the current branch. Do not use it on important work unless you understand the consequences and have a backup.

For shared history that has already been pushed, prefer `git revert` rather than rewriting history with reset.

## F. Interview Question

**Q: What is the difference between `git reset --soft`, `--mixed`, and `--hard`?**

A: They differ in what happens to the staging area and working directory after HEAD moves. Soft preserves both, mixed resets the index but preserves files, and hard resets both the index and tracked files.

## G. Exercise

In a disposable repository, create two commits and practise `--soft` and `--mixed`. Only practise `--hard` after creating a backup or confirming the repository contains no important work.


---

# 12. Git Revert

## A. What is `git revert`?

`git revert` creates a new commit that reverses the changes introduced by an earlier commit. It does not remove the original commit from history.

## B. Syntax

```bash
git revert <commit-id>
```

To prepare the inverse changes without committing immediately:

```bash
git revert --no-commit <commit-id>
```

## C. Practical Example

First inspect recent history:

```bash
git log --oneline
```

Choose the commit you want to reverse, then run:

```bash
git revert <commit-id>
```

Git may open an editor for the revert commit message. Save and close the editor to complete the revert.

## D. Expected Result

- The original commit remains in history.
- Git creates a new commit that reverses the selected commit's changes.
- The working tree should be clean after a successful revert commit.

If conflicts occur, resolve them, stage the resolved files, and continue the revert:

```bash
git add <resolved-file>
git revert --continue
```

To cancel an in-progress revert:

```bash
git revert --abort
```

## E. Reset vs Revert

| `git reset` | `git revert` |
|---|---|
| Moves the branch reference | Adds a new commit |
| Can rewrite local history | Preserves existing history |
| Often used for local commit changes | Safer for commits already shared |

## F. Interview Question

**Q: Why is `git revert` generally safer than `git reset` for shared commits?**

A: Revert adds a new commit instead of moving a shared branch backwards, so collaborators keep the existing history.

## G. Exercise

1. Create a commit that adds a sample file.
2. Note its commit ID.
3. Run `git revert <commit-id>`.
4. Inspect `git log --oneline` and `git status`.


---

# 13. Git Clean

## A. What is `git clean`?

`git clean` removes untracked files from the working directory. It does not remove files that are already tracked by Git.

## B. Preview Before Deleting

Always preview the files first:

```bash
git clean -n
```

The `-n` option means dry run. It lists what would be removed without deleting anything.

## C. Common Commands

Remove untracked files:

```bash
git clean -f
```

Remove untracked files and untracked directories:

```bash
git clean -fd
```

Preview files and directories:

```bash
git clean -nd
```

Include ignored files in the cleanup:

```bash
git clean -fdX
```

Be especially careful with ignored files: they may include local configuration, environment files, or generated data.

## D. Safe Practical Workflow

```bash
git status
git clean -nd
```

Review every listed path. Only if you are certain those files and directories can be deleted, run:

```bash
git clean -fd
git status
```

## E. Important Warning

`git clean -f` and `git clean -fd` permanently delete matching untracked files and directories. They do not move them to the Recycle Bin. Never run them without reviewing the dry-run output.

## F. Interview Question

**Q: What is the purpose of `git clean -n`?**

A: It previews which untracked files would be removed, without deleting them.

## G. Exercise

1. Create an untracked temporary file in a disposable repository.
2. Run `git clean -n`.
3. Confirm the file is listed.
4. Only if it is safe to delete, run `git clean -f`.


---

# Troubleshooting

| Problem | What to check |
|---|---|
| `git rm` says pathspec did not match | Check the exact file path and spelling |
| `git mv` destination directory missing | Create the destination directory first |
| `git restore` discarded edits | Check `git status`, `git diff`, and available commits; uncommitted discarded work may not be recoverable |
| `git reset` moved the branch unexpectedly | Inspect `git log --oneline --decorate` and `git reflog` |
| `git revert` reports conflicts | Resolve conflicts, stage resolved files, then use `git revert --continue` |
| `git clean` preview lists important files | Do not run the force command; review the paths carefully |

# Revision Checklist

- [ ] I can remove tracked files with `git rm`.
- [ ] I can rename or move tracked files with `git mv`.
- [ ] I understand how to discard or unstage changes with `git restore`.
- [ ] I can explain the three main `git reset` modes.
- [ ] I can reverse a commit with `git revert`.
- [ ] I know how to preview untracked files with `git clean -n`.

# Summary

This notebook covered six commands for managing files and undoing changes:

- `git rm` — remove tracked files.
- `git mv` — move or rename tracked files.
- `git restore` — restore file contents or unstage changes.
- `git reset` — move the current branch reference and optionally reset the index and working tree.
- `git revert` — reverse a commit by creating a new commit.
- `git clean` — remove untracked files after previewing them.

**Safety rule:** Inspect status and diffs before undoing changes. Use disposable repositories for destructive-command practice.
